# MobileNetV2 Flood Detector - Balanced Training

**Phase 2 Improvement:** Train on balanced 50/50 flood/not-flood dataset.

**Expected Improvement:** Reduced false positives, better specificity.

**Runtime:** Use GPU (Runtime > Change runtime type > GPU)

## 1. Setup Kaggle API

In [ ]:
!pip install -q kaggle

from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 2. Download Datasets

In [ ]:
# Flood dataset
!kaggle datasets download -d faizalkarim/flood-area-segmentation
!unzip -q flood-area-segmentation.zip -d flood_data/

# Non-flood scenes (Places365 mini or generate synthetic)
# Option A: Use Places365 subset
# !wget http://data.csail.mit.edu/places/places365/val_256.tar

# Option B: Generate synthetic (simpler, faster)
print("Generating synthetic non-flood images...")

In [ ]:
import os
import random
from PIL import Image, ImageDraw
from pathlib import Path

# Create directories
Path("data/flood").mkdir(parents=True, exist_ok=True)
Path("data/not_flood").mkdir(parents=True, exist_ok=True)

def generate_synthetic_negatives(count, output_dir):
    """Generate diverse non-flood images."""
    categories = ["urban", "grass", "desert", "forest", "road"]
    
    for i in range(count):
        category = random.choice(categories)
        img = Image.new('RGB', (256, 256))
        draw = ImageDraw.Draw(img)
        
        if category == "urban":
            base = (random.randint(80, 140), random.randint(80, 140), random.randint(80, 140))
            img.paste(base, [0, 0, 256, 256])
            for _ in range(random.randint(5, 20)):
                x, y = random.randint(0, 200), random.randint(0, 200)
                w, h = random.randint(20, 60), random.randint(30, 80)
                shade = random.randint(-30, 30)
                color = tuple(max(0, min(255, c + shade)) for c in base)
                draw.rectangle([x, y, x+w, y+h], fill=color, outline=(50, 50, 50))
                
        elif category == "grass":
            base = (random.randint(40, 80), random.randint(120, 180), random.randint(30, 70))
            img.paste(base, [0, 0, 256, 256])
            for _ in range(300):
                x, y = random.randint(0, 255), random.randint(0, 255)
                shade = random.randint(-20, 20)
                color = tuple(max(0, min(255, c + shade)) for c in base)
                draw.line([x, y, x + random.randint(-2, 2), y + random.randint(5, 12)], fill=color)
                
        elif category == "desert":
            base = (random.randint(180, 230), random.randint(150, 200), random.randint(100, 160))
            img.paste(base, [0, 0, 256, 256])
            for _ in range(150):
                x, y = random.randint(0, 255), random.randint(0, 255)
                size = random.randint(2, 6)
                shade = random.randint(-15, 15)
                color = tuple(max(0, min(255, c + shade)) for c in base)
                draw.ellipse([x, y, x+size, y+size], fill=color)
                
        elif category == "forest":
            base = (random.randint(20, 50), random.randint(60, 100), random.randint(20, 50))
            img.paste(base, [0, 0, 256, 256])
            for _ in range(60):
                x, y = random.randint(0, 230), random.randint(0, 230)
                size = random.randint(10, 35)
                shade = random.randint(-20, 20)
                color = tuple(max(0, min(255, c + shade)) for c in base)
                draw.ellipse([x, y, x+size, y+size], fill=color)
                
        elif category == "road":
            img.paste((100, 100, 100), [0, 0, 256, 256])
            draw.rectangle([110, 0, 146, 256], fill=(80, 80, 80))
            for y in range(0, 256, 40):
                draw.rectangle([125, y, 131, y+20], fill=(255, 255, 255))
        
        img.save(f"{output_dir}/synthetic_{category}_{i:04d}.jpg")
        
    print(f"Generated {count} synthetic images")

# Generate non-flood images
generate_synthetic_negatives(1000, "data/not_flood")

In [ ]:
# Copy flood images from Kaggle dataset
import shutil
from pathlib import Path

flood_source = list(Path("flood_data").rglob("*.jpg")) + list(Path("flood_data").rglob("*.png"))
print(f"Found {len(flood_source)} flood images")

for i, img in enumerate(flood_source[:1000]):
    shutil.copy(img, f"data/flood/flood_{i:04d}.jpg")

print(f"Copied {min(len(flood_source), 1000)} flood images")

## 3. Create Balanced Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image
from pathlib import Path
import random

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

class BalancedFloodDataset(Dataset):
    def __init__(self, flood_dir, not_flood_dir, transform=None, balance=True):
        self.transform = transform
        
        flood_images = list(Path(flood_dir).glob("*.jpg"))
        not_flood_images = list(Path(not_flood_dir).glob("*.jpg"))
        
        # Balance: use same number from each class
        if balance:
            min_count = min(len(flood_images), len(not_flood_images))
            random.shuffle(flood_images)
            random.shuffle(not_flood_images)
            flood_images = flood_images[:min_count]
            not_flood_images = not_flood_images[:min_count]
        
        self.images = [(p, 1) for p in flood_images] + [(p, 0) for p in not_flood_images]
        random.shuffle(self.images)
        
        print(f"Dataset: {len(flood_images)} flood + {len(not_flood_images)} not-flood = {len(self.images)} total")
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        path, label = self.images[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Create datasets
full_dataset = BalancedFloodDataset("data/flood", "data/not_flood", transform=train_transform)

# Split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## 4. Model Setup

In [ ]:
model = models.mobilenet_v2(pretrained=True)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 1),
    nn.Sigmoid()
)

model = model.to(device)

# Freeze early layers
for param in model.features[:10].parameters():
    param.requires_grad = False

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Training

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.float().to(device)
        
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += ((outputs > 0.5) == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    tp, fp, tn, fn = 0, 0, 0, 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.float().to(device)
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            # Confusion matrix
            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return total_loss / len(loader), correct / total, precision, recall, specificity

In [ ]:
best_val_acc = 0
history = []

print("Training on balanced dataset...")
print("-" * 70)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, precision, recall, specificity = validate(model, val_loader, criterion)
    scheduler.step()
    
    history.append({
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss, 'val_acc': val_acc,
        'precision': precision, 'recall': recall, 'specificity': specificity
    })
    
    print(f"Epoch {epoch+1:2d}: Train {train_acc:.1%} | Val {val_acc:.1%} | "
          f"Precision {precision:.1%} | Recall {recall:.1%} | Specificity {specificity:.1%}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "mobilenetv2_flood_balanced.pth")
        print(f"         -> Best model saved!")

print("-" * 70)
print(f"Best validation accuracy: {best_val_acc:.1%}")

## 6. Export

In [ ]:
model.load_state_dict(torch.load("mobilenetv2_flood_balanced.pth"))
model.eval()

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'architecture': 'mobilenet_v2_balanced',
        'num_classes': 1,
        'input_size': IMG_SIZE,
        'best_val_acc': best_val_acc,
        'training': 'balanced_50_50'
    }
}, "mobilenetv2_flood_balanced_final.pth")

from google.colab import files
files.download('mobilenetv2_flood_balanced_final.pth')